# 02 - Object Detection with YOLO

## From Manual Rules to AI Perception

In Notebook 01, *we* wrote the rules — blur this, threshold that, find contours here. **YOLO** (You Only Look Once) flips this entirely: a neural network trained on millions of images learns to detect, classify, and track objects in a single pass.

YOLOv8 from [Ultralytics](https://docs.ultralytics.com/) is the model family we'll use. By simply swapping the weights file, we unlock different capabilities:

| Weights File | Task | What It Does |
|---|---|---|
| `yolov8n.pt` | **Detection** | Bounding boxes + class labels (80 COCO classes) |
| `yolov8n-pose.pt` | **Pose Estimation** | 17-keypoint skeleton on every person |
| `yolov8n-seg.pt` | **Instance Segmentation** | Pixel-level masks per object |

The `n` stands for **Nano** — the smallest, fastest variant, perfect for real-time laptop demos.

> **How to exit:** Press `q` in any OpenCV window to close it safely.

---

## Task 1: Live Object Detection & Tracking

This is the "Hello World" of modern CV. YOLO processes each webcam frame and returns:
- **Bounding boxes** around every detected object
- **Class labels** (person, chair, phone, cup, …)
- **Confidence scores** (how sure the model is)
- **Track IDs** (persistent identity across frames via `model.track()`)

We also add a simple **people counter** — counting how many objects have class ID `0` (person in the COCO dataset).

In [ ]:
import cv2
from ultralytics import YOLO

# Load the "Nano" detection model (optimised for speed on laptops)
model = YOLO("models/yolov8n.pt")

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam — is it connected?")

print("YOLO Detection & Tracking — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO with tracking — persist=True keeps track IDs stable across frames
    results = model.track(frame, persist=True, conf=0.5, verbose=False)

    # plot() draws boxes, labels, confidence scores, and track IDs automatically
    annotated = results[0].plot()

    # --- People counter overlay ---
    boxes = results[0].boxes
    people = int((boxes.cls == 0).sum()) if boxes.cls is not None else 0
    total  = len(boxes)

    cv2.putText(annotated, f"People: {people}  |  Total objects: {total}",
                (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    cv2.imshow("YOLO Detection & Tracking", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

---

## Task 2: Trajectory Tracking

YOLO's tracker assigns each object a persistent **Track ID**. By recording the centre point of each person's bounding box over time, we can draw their **movement trail** — a trajectory line that follows them across the frame.

### How It Works
1. `model.track(persist=True)` maintains identities across frames.
2. We store each track ID's `(x, y)` centre in a dictionary.
3. `cv2.polylines()` connects the dots into a coloured trail.
4. We keep only the last 30 points per track to avoid clutter.

In [ ]:
from collections import defaultdict
import numpy as np
import cv2
from ultralytics import YOLO

model = YOLO("models/yolov8n.pt")
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

# Dictionary: track_id → list of (x, y) centre points
track_history = defaultdict(list)
TRAIL_LENGTH = 30  # max points per trail (older points are dropped)

print("Trajectory Tracker — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, conf=0.5, verbose=False)
    annotated = results[0].plot()

    # Only draw trails if tracking IDs are available
    if results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.int().cpu().tolist()
        boxes_xywh = results[0].boxes.xywh.cpu()

        for (x, y, w, h), tid in zip(boxes_xywh, track_ids):
            # Record the centre of this bounding box
            track = track_history[tid]
            track.append((float(x), float(y)))

            # Trim old points so the trail doesn't grow forever
            if len(track) > TRAIL_LENGTH:
                track.pop(0)

            # Draw the trajectory as a polyline
            if len(track) >= 2:
                pts = np.array(track, dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(annotated, [pts], isClosed=False,
                              color=(0, 255, 255), thickness=3)

    cv2.imshow("YOLO Trajectory Tracker", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

---

## Task 3: Pose Estimation

By swapping to `yolov8n-pose.pt`, the exact same pipeline now detects **17 body keypoints** per person (nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles) and draws a skeleton connecting them.

This is the foundation of:
- **Gesture recognition** (wave to trigger an action)
- **Fitness tracking** (count reps by tracking joint angles)
- **Sports analytics** (measure stride length, posture)
- **Sign language interpretation**

The `plot()` function handles all the skeleton drawing automatically.

In [ ]:
import cv2
from ultralytics import YOLO

# Swap to the Pose model — same API, different weights
pose_model = YOLO("models/yolov8n-pose.pt")

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("Pose Estimation — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = pose_model(frame, conf=0.5, verbose=False)
    annotated = results[0].plot()  # draws skeletons + keypoints automatically

    # Count how many people have detected keypoints
    n_people = len(results[0].keypoints) if results[0].keypoints is not None else 0

    cv2.putText(annotated, f"Pose Estimation  |  {n_people} skeleton(s)",
                (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 100, 255), 2)

    cv2.imshow("YOLO Pose Estimation", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

---

## Task 4: Instance Segmentation

Swapping to `yolov8n-seg.pt` goes beyond bounding boxes — the model now predicts a **pixel-level mask** for every detected object. Instead of a rectangle, you get the exact silhouette.

This is called **instance segmentation** because each individual object gets its own mask (as opposed to semantic segmentation, which just labels every pixel by class without distinguishing between separate objects of the same type).

> In Notebook 03, we'll see how **SAM 2** can refine these masks to be even more precise.

In [ ]:
import cv2
from ultralytics import YOLO

# Swap to the Segmentation model — same API, different weights
seg_model = YOLO("models/yolov8n-seg.pt")

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("Instance Segmentation — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # retina_masks=True produces smoother, higher-resolution masks
    results = seg_model(frame, conf=0.5, retina_masks=True, verbose=False)
    annotated = results[0].plot()  # fills segmented pixels with colour

    n_objects = len(results[0].boxes)
    cv2.putText(annotated, f"Segmentation  |  {n_objects} object(s)",
                (15, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)

    cv2.imshow("YOLO Instance Segmentation", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

---

## Bonus: Live Face Filter (Sunglasses Overlay)

Let's combine everything into a fun demo. Using the **Pose model**, we extract the **eye keypoints** (keypoints 1 and 2) and use them to position a sunglasses image on every detected face.

### The Pipeline
1. Run YOLOv8 Pose to get 17 keypoints per person.
2. Extract the left eye and right eye coordinates.
3. Calculate the midpoint between the eyes (= where to center the glasses).
4. Calculate the distance between the eyes (= how wide to make them).
5. Overlay the resized sunglasses using a binary mask to "green-screen" out the black background.

In [ ]:
import cv2
import math
import numpy as np
from ultralytics import YOLO

# ---------- Setup ----------
pose_model = YOLO("models/yolov8n-pose.pt")

SUNGLASSES_PATH = "samples/images/sunglasses.png"
sunglasses_img = cv2.imread(SUNGLASSES_PATH)
if sunglasses_img is None:
    raise FileNotFoundError(
        f"Could not load sunglasses image at '{SUNGLASSES_PATH}'. "
        "Place a sunglasses.png (black background) in samples/images/."
    )

GLASSES_ASPECT = sunglasses_img.shape[0] / sunglasses_img.shape[1]  # h / w
EYE_SCALE = 2.5      # how much wider than eye distance the glasses should be
MIN_WIDTH = 10        # ignore tiny detections
BRIGHT_THRESH = 10    # brightness cutoff for the black-background mask


def overlay_image(background, overlay, cx, cy, size):
    """Overlay an image with a black background onto a frame using binary masking."""
    bg_h, bg_w = background.shape[:2]

    # Resize overlay to target size
    overlay = cv2.resize(overlay, size)
    oh, ow = overlay.shape[:2]

    # Calculate placement coordinates (centred on cx, cy)
    y1, y2 = int(cy - oh / 2), int(cy + oh / 2)
    x1, x2 = int(cx - ow / 2), int(cx + ow / 2)

    # Track how much of the overlay to use (for edge clipping)
    oy1, oy2, ox1, ox2 = 0, oh, 0, ow

    # Clip against frame boundaries
    if y1 < 0:    oy1 -= y1;  y1 = 0
    if x1 < 0:    ox1 -= x1;  x1 = 0
    if y2 > bg_h: oy2 -= (y2 - bg_h); y2 = bg_h
    if x2 > bg_w: ox2 -= (x2 - bg_w); x2 = bg_w

    # Skip if entirely off-screen
    if y1 >= y2 or x1 >= x2 or oy1 >= oy2 or ox1 >= ox2:
        return background

    bg_roi = background[y1:y2, x1:x2]
    fg_roi = overlay[oy1:oy2, ox1:ox2]

    # Build a binary mask: white = glasses pixels, black = background
    gray = cv2.cvtColor(fg_roi, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, BRIGHT_THRESH, 255, cv2.THRESH_BINARY)
    mask_inv = cv2.bitwise_not(mask)

    # Composite: cut a hole in the background, paste glasses into it
    bg_part = cv2.bitwise_and(bg_roi, bg_roi, mask=mask_inv)
    fg_part = cv2.bitwise_and(fg_roi, fg_roi, mask=mask)
    background[y1:y2, x1:x2] = cv2.add(bg_part, fg_part)

    return background


# ---------- Webcam loop ----------
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("Face Filter — press 'q' to quit.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = pose_model(frame, conf=0.5, verbose=False)
    output = frame.copy()  # draw on a clean copy (no skeleton lines)

    if results[0].keypoints is not None and results[0].keypoints.xy is not None:
        for kps in results[0].keypoints.xy:
            if len(kps) < 3:
                continue

            left_eye, right_eye = kps[1], kps[2]

            # Skip if either eye was not detected (coordinates at origin)
            if left_eye[0] == 0 or right_eye[0] == 0:
                continue

            # Position: midpoint between the eyes
            cx = int((left_eye[0] + right_eye[0]) / 2)
            cy = int((left_eye[1] + right_eye[1]) / 2)

            # Size: proportional to the distance between the eyes
            eye_dist = math.hypot(
                float(right_eye[0] - left_eye[0]),
                float(right_eye[1] - left_eye[1]),
            )
            g_width = int(eye_dist * EYE_SCALE)

            if g_width > MIN_WIDTH:
                g_height = int(g_width * GLASSES_ASPECT)
                output = overlay_image(output, sunglasses_img, cx, cy,
                                       (g_width, g_height))

    cv2.putText(output, "YOLO Face Filter", (15, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)

    cv2.imshow("Virtual Sunglasses", output)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

---

## Recap

| What We Did | Weights | Key Takeaway |
|---|---|---|
| **Object Detection & Tracking** | `yolov8n.pt` | Bounding boxes + class labels + persistent track IDs |
| **Trajectory Tracking** | `yolov8n.pt` | Record centre points over time and draw trails |
| **Pose Estimation** | `yolov8n-pose.pt` | 17-keypoint skeleton per person |
| **Instance Segmentation** | `yolov8n-seg.pt` | Pixel-level masks per object |
| **Face Filter** | `yolov8n-pose.pt` | Use keypoints to position overlays in real time |

### Key Insight
Every task above used the **same webcam loop pattern** from Notebook 01. The only thing that changed was the model weights file. That's the power of a unified framework like Ultralytics — swap one file and unlock completely different AI capabilities.

### What's Next?
In Notebook 03, we move into **Foundation Models**. SAM 2 will give us pixel-perfect masks far beyond what YOLO-Seg can achieve, and CLIP will classify images using plain English — no training required.